In [1]:
import os
import numpy as np
import pandas as pd
import math
from scipy import stats


ref_genome_folder = '../exp_data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY','chrMT']
resolution = 50000


# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)

chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


### convert residual into .bw file

In [10]:
bin_dir = '../CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/b50000'

bin_dict = dict()
for key in chr_bin_number.keys():
    bin_dict[key] = np.zeros(chr_bin_number[key])

keys = list(chr_bin_number.keys())
key_num = 0
for file in os.listdir(bin_dir):
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            start_, end_, obs, exp ,var = line.strip().split()
            residual = float(obs) - float(exp)
            bin_ = math.floor(int(start_)/resolution)
            bin_dict[keys[key_num]][bin_] = residual
    key_num = key_num +1

In [11]:
output_path = 'CNV/hic_residual.bw'
with open(output_path, 'w+') as f:
    for key in bin_dict.keys():
        #percentile_99, percentile_1 = np.percentile(bin_dict[key], 99), np.percentile(bin_dict[key], 1)
        for bin_, residual in enumerate(bin_dict[key]):
            bin_start = bin_ * 50000
            bin_end = (bin_+1) * 50000 if (bin_+1) * 50000 <= chr_length[key] else chr_length[key]
            #residual = 0 if (residual >= percentile_99) or (residual <= percentile_1) else residual
            residual = round(residual,4)
            f.write(f'{key}\t{bin_start}\t{bin_end}\t{residual}\n')

### convert hic copy ratio into .bw

In [4]:
bin_dir = '../CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/b50000'

bin_dict = dict()
for key in chr_bin_number.keys():
    bin_dict[key] = np.zeros(chr_bin_number[key])

keys = list(chr_bin_number.keys())
key_num = 0
for file in os.listdir(bin_dir):
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            start_, end_, obs, exp ,var = line.strip().split()
            bin_ = math.floor(int(start_)/resolution)
            log2_ratio = math.log2(float(obs) / float(exp)) if not ((float(exp) == 0) or (float(obs) == 0)) else 0
            bin_dict[keys[key_num]][bin_] = log2_ratio
    key_num = key_num +1

In [5]:
output_path = 'CNV/hic_log2_copy_ratio.bw'
with open(output_path, 'w+') as f:
    for key in bin_dict.keys():
        percentile_99, percentile_1 = np.percentile(bin_dict[key], 99), np.percentile(bin_dict[key], 1)
        for bin_, log2_ratio in enumerate(bin_dict[key]):
            bin_start = bin_ * 50000
            bin_end = (bin_+1) * 50000 if (bin_+1) * 50000 <= chr_length[key] else chr_length[key]
            log2_ratio = 0 if (log2_ratio >= percentile_99) or (log2_ratio <= percentile_1) else log2_ratio
            log2_ratio = round(log2_ratio,4)
            f.write(f'{key}\t{bin_start}\t{bin_end}\t{log2_ratio}\n')

### convert .bin into .bw

In [6]:
bin_dir = '../CNV/K562_WGS_CNV/bin'

bin_dict = dict()
for key in chr_bin_number.keys():
    bin_dict[key] = np.zeros(chr_bin_number[key])

for file in os.listdir(bin_dir):
    if file[0] == '.':
        continue
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            chr_ = file.split('.')[0]
            start_, end_, obs, exp, var = line.strip().split()
            bin_ = math.floor(int(start_)/50000)
            log2_ratio = math.log2(float(obs) / float(exp)) if not ((float(exp) == 0) or (float(obs) == 0)) else 0
            bin_dict[chr_][bin_] = log2_ratio


In [8]:
bw_file = open('CNV/WGS_log2_copy_ratio.bw', 'w+')


for chr_ in bin_dict.keys():
    upper, lower = np.percentile(bin_dict[chr_], 99), np.percentile(bin_dict[chr_], 1)
    for bin_, log2_ratio in enumerate(bin_dict[chr_]):
        log2_ratio = 0.0 if (log2_ratio >= upper) or (log2_ratio <= lower) else log2_ratio
        start_ = bin_ * 50000
        end_ = (bin_+1) * 50000 if not ((bin_+1) * 50000 > chr_length[chr_]) else chr_length[chr_]

        log2_ratio = round(log2_ratio,4)
        bw_file.write(f'{chr_}\t{start_}\t{end_}\t{log2_ratio}\n')
bw_file.close()